# 06 - Policy Improvement — Paper & Pencil Exercises

# Exercises — Policy Improvement (Model-Free Control)

These paper-and-pencil exercises reinforce Chapter 06: the SARSA (on-policy) and Q-learning (off-policy) update targets, maximisation bias and its correction by double learning, the GLIE conditions for convergence, and eligibility traces (accumulating vs replacing) for SARSA($\lambda$). Notation: $Q(s,a)$ is the estimate of $q_\pi(s,a)$; the reward for acting at $t$ is $R_{t+1}$.

---


## Exercise 6.1 — SARSA vs Q-learning: one update

**Problem.** The agent is in state $s$, takes action $a$, receives reward $R=0$, and lands in $s'$. Current estimates: $Q(s,a)=0.5$ and $Q(s',\cdot)=(0.4,\,0.9)$ for the two actions in $s'$. Parameters: $\gamma=0.9,\ \alpha=0.1$. The $\varepsilon$-greedy behaviour policy happens to select the **exploratory** next action $a'$ with $Q(s',a')=0.4$ (not the greedy one).

Compute the updated $Q(s,a)$ under (i) SARSA and (ii) Q-learning, and explain the difference.


**Solution.**

**Step 1 — SARSA (on-policy).** The target uses the value of the action **actually taken** next, $Q(s',a')$:

$$\text{target}_{\text{SARSA}} = R + \gamma\,Q(s',a') = 0 + 0.9(0.4) = 0.36,$$
$$Q(s,a) \leftarrow 0.5 + 0.1\,(0.36 - 0.5) = 0.5 + 0.1(-0.14) = 0.486.$$

**Step 2 — Q-learning (off-policy).** The target uses the value of the **greedy** next action, $\max_{a'}Q(s',a')=0.9$, regardless of what the behaviour policy does:

$$\text{target}_{\text{QL}} = R + \gamma\max_{a'}Q(s',a') = 0 + 0.9(0.9) = 0.81,$$
$$Q(s,a) \leftarrow 0.5 + 0.1\,(0.81 - 0.5) = 0.5 + 0.1(0.31) = 0.531.$$

**Step 3 — Difference.** SARSA moves $Q(s,a)$ *down* to $0.486$ (it accounts for the exploratory, lower-valued action the agent will actually take), while Q-learning moves it *up* to $0.531$ (it assumes greedy continuation). The only difference is which next-state action supplies the bootstrap value.

> **Key concept.** SARSA is **on-policy**: it evaluates the policy it actually follows (including exploration). Q-learning is **off-policy**: it learns the value of the *greedy* target policy while following an exploratory behaviour policy. This is why SARSA tends to learn safer behaviour when exploration is risky.


## Exercise 6.2 — Maximisation bias

**Problem.** In some state, two actions both have **true** value $q(a_1)=q(a_2)=0$. Because of noise, the current *estimates* are independent random variables each equal to $+1$ or $-1$ with probability $\tfrac12$. Q-learning would use $\max_a Q(a)$ as an estimate of $\max_a q(a)=0$.

Compute $\mathbb{E}[\max(Q(a_1),Q(a_2))]$ and explain why this demonstrates a systematic *positive* (maximisation) bias.


**Solution.**

**Step 1 — Enumerate the four equally likely outcomes** of $(Q(a_1),Q(a_2))$:

| $Q(a_1)$ | $Q(a_2)$ | prob. | $\max$ |
|---|---|---|---|
| $+1$ | $+1$ | $1/4$ | $+1$ |
| $+1$ | $-1$ | $1/4$ | $+1$ |
| $-1$ | $+1$ | $1/4$ | $+1$ |
| $-1$ | $-1$ | $1/4$ | $-1$ |

**Step 2 — Expected maximum.**

$$\mathbb{E}[\max] = \tfrac34(+1) + \tfrac14(-1) = 0.5.$$

**Step 3 — Compare with the truth.** The true quantity is $\max_a q(a)=0$, but the *estimated* maximum has expectation $+0.5$. So

$$\mathbb{E}[\max_a Q(a)] = 0.5 \;>\; 0 = \max_a q(a).$$

The estimator is **biased upward**: taking the maximum of noisy estimates systematically overestimates the true maximum, because the $\max$ operator preferentially selects whichever estimate happened to be inflated by noise.

> **Key concept.** Q-learning uses "the **max of estimates** as an estimate of the max", which is optimistically biased. **Double Q-learning** removes this by using one estimator to *select* the best action and a second, independent estimator to *evaluate* it — decoupling selection from evaluation (next exercise).


## Exercise 6.3 — A double Q-learning update

**Problem.** Double Q-learning keeps two tables $Q_1, Q_2$. On a step with reward $R=0$, $\gamma=0.9$, $\alpha=0.1$, the transition reaches $s'$ where $Q_1(s',\cdot)=(0.2,\,0.8)$ and $Q_2(s',\cdot)=(0.5,\,0.3)$. Suppose the coin flip selects $Q_1$ to be updated, and $Q_1(s,a)=0.5$.

Perform the double Q-learning update of $Q_1(s,a)$, and compare with what plain Q-learning would have used.


**Solution.**

**Step 1 — Select the action with $Q_1$** (the estimator being updated chooses the argmax):

$$a^* = \arg\max_{a'} Q_1(s',a') = a_2 \quad(\text{since } 0.8 > 0.2).$$

**Step 2 — Evaluate that action with the *other* table $Q_2$:**

$$\text{target} = R + \gamma\,Q_2(s',a^*) = 0 + 0.9\,(0.3) = 0.27.$$

**Step 3 — Update $Q_1(s,a)$:**

$$Q_1(s,a) \leftarrow 0.5 + 0.1\,(0.27 - 0.5) = 0.5 + 0.1(-0.23) = 0.477.$$

**Step 4 — Compare with plain Q-learning.** Plain Q-learning would both select *and* evaluate with the same table: target $= R + \gamma\max_{a'}Q_1(s',a') = 0.9(0.8) = 0.72$ — much larger. The double estimator's evaluation ($Q_2$ says the $Q_1$-preferred action is only worth $0.3$) **counteracts the optimistic bias**.

> **Key concept.** By asking one estimator "which action is best?" and a *different* one "how good is it?", double Q-learning breaks the correlation that causes maximisation bias, at the cost of learning two tables (each from half the data).


## Exercise 6.4 — GLIE conditions for convergence to the optimal policy

**Problem.** For on-policy control (MC control, SARSA) to converge to the optimal policy, the exploration must be **GLIE** (Greedy in the Limit with Infinite Exploration): (i) every state–action pair is visited infinitely often, and (ii) the policy becomes greedy in the limit ($\varepsilon_t\to 0$). Using an $\varepsilon$-greedy policy with schedule $\varepsilon_t$, classify each schedule as GLIE or not:

$$\text{(a)}\ \varepsilon_t=\tfrac1t, \qquad \text{(b)}\ \varepsilon_t=0.1\ (\text{constant}), \qquad \text{(c)}\ \varepsilon_t=\tfrac{1}{t^2}.$$


**Solution.** The random-exploration probability per step is proportional to $\varepsilon_t$, so "infinite exploration" requires $\sum_t \varepsilon_t = \infty$, and "greedy in the limit" requires $\varepsilon_t\to 0$.

**(a) $\varepsilon_t = 1/t$.**
- $\varepsilon_t\to 0$ ✓ (greedy in the limit).
- $\sum_t 1/t = \infty$ (harmonic series) ✓ (infinite exploration — every action keeps being tried).
- **GLIE ✓** → convergence to the optimal policy is guaranteed.

**(b) $\varepsilon_t = 0.1$ (constant).**
- $\varepsilon_t \not\to 0$ ✗ — the policy stays $0.1$-exploratory forever.
- Infinite exploration holds ($\sum 0.1=\infty$), but condition (ii) fails.
- **Not GLIE** → converges to the best $\varepsilon$-greedy policy, *not* the optimal greedy policy.

**(c) $\varepsilon_t = 1/t^2$.**
- $\varepsilon_t\to 0$ ✓.
- $\sum_t 1/t^2 < \infty$ (converges) ✗ — total exploration is finite, so some actions may be tried only finitely often.
- **Not GLIE** → condition (i) fails; convergence not guaranteed.

| schedule | $\varepsilon_t\to0$? | $\sum\varepsilon_t=\infty$? | GLIE? |
|---|---|---|---|
| $1/t$ | ✓ | ✓ | **yes** |
| $0.1$ | ✗ | ✓ | no |
| $1/t^2$ | ✓ | ✗ | no |

> **Key concept.** GLIE balances two opposing needs: keep exploring *enough* to see every option infinitely often ($\sum\varepsilon_t=\infty$), yet decay exploration to zero so the policy becomes optimal in the limit ($\varepsilon_t\to0$). Only $\varepsilon_t=1/t$ (decay neither too fast nor too slow) satisfies both — the direct control analogue of the step-size conditions in Chapter 04.


## Exercise 6.5 — Accumulating vs replacing eligibility traces

**Problem.** In SARSA($\lambda$) the eligibility trace of a state–action pair is incremented when the pair is visited and decayed by $\gamma\lambda$ each step. Take $\gamma=1,\ \lambda=0.5$ (so the decay factor is $\gamma\lambda=0.5$). A particular pair $(s,a)$ is visited at time steps $t=0$ and $t=2$ (and not at $t=1$). Track its eligibility trace $E(s,a)$ through the update at $t=2$ under

- **accumulating** traces: $E \leftarrow E + 1$ on a visit;
- **replacing** traces: $E \leftarrow 1$ on a visit (clip to $1$).

Assume each step performs *visit-then-decay*.


**Solution.** Decay factor $\gamma\lambda = 0.5$. We apply, at each step: first the increment (if visited), then multiply by $0.5$.

**Accumulating traces:**

| step | visited? | after increment | after decay ($\times0.5$) |
|---|---|---|---|
| $t=0$ | yes | $0+1=1$ | $0.5$ |
| $t=1$ | no | $0.5$ | $0.25$ |
| $t=2$ | yes | $0.25+1=1.25$ | $0.625$ |

**Replacing traces:**

| step | visited? | after set/clip | after decay ($\times0.5$) |
|---|---|---|---|
| $t=0$ | yes | $1$ | $0.5$ |
| $t=1$ | no | $0.25$ (no visit) | $0.25$ |
| $t=2$ | yes | $1$ (reset) | $0.5$ |

**Comparison.** At the second visit, the accumulating trace reaches $1.25$ (it *adds* to the residual $0.25$), whereas the replacing trace is reset to $1$. So after $t=2$ the accumulating trace ($0.625$) exceeds the replacing one ($0.5$).

**Interpretation.** Accumulating traces can *exaggerate* the credit assigned to frequently revisited pairs (their trace can exceed $1$); replacing traces cap the credit at $1$, which moderates the influence of frequent events and often helps recent, rarer events surface. Both propagate the TD error backward to *all* recently-visited pairs, weighted by their trace.

> **Key concept.** Eligibility traces implement the **backward view** of $\lambda$-returns: instead of waiting to compute $G_t^\lambda$, the current TD error updates every eligible past pair in proportion to its trace. Accumulating vs replacing is a choice about how to credit repeated visits.
